# CIFAR-10 Image Classification using a Convolutional Neural Network (CNN)

A PyTorch implementation of a Convolutional Neural Network trained from scratch to classify images from the **CIFAR-10** dataset into 10 categories.

##  Project Overview
This project builds and trains a custom CNN architecture on the CIFAR-10 dataset, achieving strong classification accuracy without using any pretrained weights. It's a hands on demonstration of core deep learning concepts: convolutional layers, pooling, activation functions, loss computation, backpropagation, and model evaluation.

##  Dataset
[CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) consists of 60,000 32x32 color images across 10 classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck):
- 50,000 training images
- 10,000 test images

##  Model Architecture
A custom CNN with 3 convolutional blocks followed by 2 fully connected layers:

| Layer | Details |
|---|---|
| Conv Block 1 | Conv2d(3→32, 3x3) → ReLU → MaxPool(2x2) |
| Conv Block 2 | Conv2d(32→64, 3x3) → ReLU → MaxPool(2x2) |
| Conv Block 3 | Conv2d(64→128, 3x3) → ReLU → MaxPool(2x2) |
| FC Layer 1 | Linear(2048 → 256) → ReLU |
| FC Layer 2 | Linear(256 → 10) |

##  Training Configuration
- **Loss Function:** Cross Entropy Loss
- **Optimizer:** Adam (default learning rate)
- **Batch Size:** 64
- **Epochs:** 10

##  Results
Achieved **~75.6% test accuracy** after 10 epochs of training.

## 🛠️ Tech Stack
- Python
- PyTorch & Torchvision

## 🚀 How to Run
1. Clone this repository
2. Install dependencies: `pip install torch torchvision`
3. Run the notebook cells in order CIFAR-10 will be downloaded automatically on first run

---


In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

## 1. Data Loading & Preprocessing

We load the CIFAR-10 dataset using `torchvision.datasets` and apply the following transformations:
- **`ToTensor()`** : converts PIL images to PyTorch tensors and scales pixel values to `[0, 1]`
- **`Normalize()`** : normalizes each channel to the range `[-1, 1]` using mean and std of `0.5`

The dataset is downloaded automatically to `./data` if not already present.


In [2]:
# DATASETS AND DATALOADER
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10

# IMAGE TRANSFORMATION => scale(0,1) & normalize(-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

C:\Users\Hp\.conda\envs\prime\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

### DataLoader Setup

We wrap the datasets in `DataLoader` objects to enable efficient batching and shuffling during training.
- **Batch size:** 64
- **Shuffle:** enabled for training data (helps generalization), disabled for test data


In [4]:
trainloader = DataLoader(trainset , batch_size = 64 , shuffle = True)
testloader = DataLoader(testset , batch_size=64)

## 2. Model Architecture — Building the CNN

We define a custom `CNN` class using `nn.Module`. The network consists of:

1. **Convolutional feature extractor** (`conv_layers`): three blocks of `Conv2d → ReLU → MaxPool2d`, progressively increasing the number of channels (3 → 32 → 64 → 128) while reducing spatial dimensions (32x32 → 16x16 → 8x8 → 4x4).
2. **Fully connected classifier** (`fc_layers`): flattens the final feature maps (128 channels × 4 × 4) and maps them to 10 output classes through a hidden layer of size 256.


In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # kernel size=2, stride=2
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )
        
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # flattening
        x = self.fc_layers(x)
        return x

In [6]:
model = CNN()

### Loss Function & Optimizer

- **Loss:** `CrossEntropyLoss` : standard choice for multi-class classification; combines `LogSoftmax` and `NLLLoss` internally.
- **Optimizer:** `Adam` : adaptive learning rate optimizer, generally converges faster and more reliably than plain SGD for this kind of task.


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## 3. Training the CNN

For each epoch, we iterate over the training data in mini batches and perform the standard PyTorch training loop:

1. Zero out gradients from the previous step
2. Forward pass : compute predictions
3. Compute the loss against true labels
4. Backpropagate gradients (`loss.backward()`)
5. Update model weights (`optimizer.step()`)

The average loss per epoch is printed to track convergence over **10 epochs**.


In [8]:
epochs = 10
for epoch in range(epochs):
    epoch_training_loss = 0.0
    
    for images, labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss += loss.item()
        
    # Move this print OUTSIDE the inner batch loop
    print(f"epoch= {epoch + 1} / {epochs} & loss= {epoch_training_loss / len(trainloader)}", flush=True)

epoch= 1 / 10 & loss= 1.3823971785700229
epoch= 2 / 10 & loss= 0.9417563927600451
epoch= 3 / 10 & loss= 0.754952850511007
epoch= 4 / 10 & loss= 0.6231833413014631
epoch= 5 / 10 & loss= 0.5177008974963747
epoch= 6 / 10 & loss= 0.4302711361623786
epoch= 7 / 10 & loss= 0.3496073450502532
epoch= 8 / 10 & loss= 0.27239751302258436
epoch= 9 / 10 & loss= 0.21430216865409213
epoch= 10 / 10 & loss= 0.1739481863187021


## 4. Evaluating the Model

We switch the model to evaluation mode (`model.eval()`) and disable gradient tracking (`torch.no_grad()`) to speed up inference and reduce memory usage. We then compute overall **test accuracy** by comparing predicted labels against ground truth over the entire test set.


In [9]:
correct_labels = 0
total_labels =0
model.eval()
with torch.no_grad():
    for images , labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs,1)
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
print(f"accuracy = {correct_labels /total_labels *100}")
        

accuracy = 75.61


## 5. Conclusion & Next Steps

The CNN achieved **~75.6% test accuracy** on CIFAR-10 after 10 epochs of training from scratch a solid baseline result for a relatively simple architecture with no data augmentation or regularization.

### Possible Improvements
- Add **data augmentation** (random crop, horizontal flip) to reduce overfitting
- Add **Batch Normalization** after conv layers for faster, more stable training
- Add **Dropout** in the fully connected layers
- Use a **learning rate scheduler**
- Experiment with deeper architectures (e.g. ResNet style blocks) or transfer learning
- Track train/test accuracy per epoch and plot loss curves for better diagnostics

---
*This notebook was built as a minor project to demonstrate a complete PyTorch deep learning pipeline: data loading, model design, training, and evaluation.*
